## Czym są GAN-y?

[Generative Adversarial Networks](https://arxiv.org/abs/1406.2661) (GAN-y) to jedna z najbardziej interesujących koncepcji we współczesnej informatyce. Dwa modele są trenowane jednocześnie w procesie rywalizacyjnym. *Generator* („artysta”) uczy się tworzyć obrazy wyglądające realistycznie, podczas gdy *dyskryminator* („krytyk sztuki”) uczy się odróżniać prawdziwe obrazy od fałszywych.

![Schemat generatora i dyskryminatora](https://github.com/tensorflow/docs/blob/master/site/en/tutorials/generative/images/gan1.png?raw=1)

Podczas treningu *generator* stopniowo coraz lepiej tworzy realistyczne obrazy, natomiast *dyskryminator* coraz skuteczniej je rozróżnia. Proces osiąga równowagę w momencie, gdy *dyskryminator* nie jest już w stanie odróżnić obrazów prawdziwych od wygenerowanych.

![Drugi schemat generatora i dyskryminatora](https://github.com/tensorflow/docs/blob/master/site/en/tutorials/generative/images/gan2.png?raw=1)


In [ ]:
# To generate GIFs
!pip install imageio
!pip install git+https://github.com/tensorflow/docs

In [ ]:
import glob
import imageio
import matplotlib.pyplot as plt
from functools import partial
import numpy as np
import os
import PIL
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
import time
import tensorflow as tf
import gdown
from zipfile import ZipFile

from IPython import display

## Generacja twarzy z użyciem sieci GAN i danych celeb.

Pobieranie danych. 

To jest duży zbiór danych, nie będziemy używać całego zbioru do treningu bo nie mamy wystarczających zasobów.

Do póżniejszej części zadania można usunąć ten plik z pamięci.

In [ ]:
os.makedirs("celeba_gan")

url = "https://drive.google.com/uc?id=1O7m1010EJjLE5QxLZiM9Fpjs7Oj6e684"
output = "celeba_gan/data.zip"
gdown.download(url, output, quiet=True)

with ZipFile("celeba_gan/data.zip", "r") as zipobj:
    zipobj.extractall("celeba_gan")


Stworzenie obiektu Dataset do uczenia w Tensorflow.

In [ ]:
dataset = keras.utils.image_dataset_from_directory(
    "celeba_gan", label_mode=None, image_size=(64, 64), batch_size=64
)
dataset = dataset.map(lambda x: x / 255.0)


Stworzenie próbki danych treningowych jako część oryginalnego zbioru danych.

In [ ]:
dim = lambda x: x[:640, ...]

train_dataset = dataset.map(dim)

del dataset

Wizualizacja przykładowej twarzy z zbioru danych

In [ ]:
for x in train_dataset:
    plt.axis("off")
    plt.imshow((x.numpy() * 255).astype("int32")[0])
    break


### Model generatora

Generator jest odpowiedzialny za tworzenie nowych, syntetycznych obrazów na podstawie losowego wektora wejściowego (*noise*). Na wejściu otrzymuje wektor liczb rzeczywistych (np. o wymiarze 100), który można interpretować jako próbkę z rozkładu losowego (najczęściej normalnego).

Architektura generatora składa się z kilku etapów:

- **Warstwa gęsta (Dense)** przekształca wektor wejściowy do wysokowymiarowej reprezentacji (tutaj: 16×16×128).
- **Batch Normalization** stabilizuje proces uczenia poprzez normalizację aktywacji.
- **LeakyReLU** wprowadza nieliniowość, zapobiegając problemowi zanikania gradientu.
- **Reshape** zmienia wektor na tensor 3D, który można interpretować jako mapę cech (feature map).

Zwykle, w dalszych warstwach (nie pokazanych tutaj) zwykle stosuje się warstwy konwolucyjne transponowane (*Conv2DTranspose*), które stopniowo zwiększają rozdzielczość obrazu.

Celem generatora jest nauczenie się takiej transformacji, aby wygenerowane obrazy były nieodróżnialne od rzeczywistych.

---

In [ ]:
def make_generator_model():
    model = tf.keras.Sequential()
    model.add(layers.Dense(16*16*128, use_bias=False, input_shape=(100,)))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    model.add(layers.Reshape((16, 16, 128)))
    assert model.output_shape == (None, 16, 16, 128)  # Note: None is the batch size

    model.add(layers.Conv2DTranspose(128, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    assert model.output_shape == (None, 32, 32, 128)
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    model.add(layers.Conv2DTranspose(3, (5, 5), strides=(2, 2), padding='same', use_bias=False, activation='tanh'))
    assert model.output_shape == (None, 64, 64, 3)

    return model

Przykładowy obraz z generatora przed treningiem.

In [ ]:
generator = make_generator_model()

noise = tf.random.normal([1, 100])
generated_image = generator(noise, training=False)

plt.imshow(generated_image[0, :, :, 0])


### Dyskryminator

Dyskryminator pełni rolę klasyfikatora binarnego — jego zadaniem jest określenie, czy dany obraz jest prawdziwy (pochodzi z danych treningowych), czy został wygenerowany przez generator.

Architektura dyskryminatora:

- **Warstwy konwolucyjne (Conv2D)** wyodrębniają cechy z obrazu (np. krawędzie, tekstury).
- **LeakyReLU** zapewnia nieliniowość.
- **Dropout** działa jako regularyzacja, zmniejszając ryzyko przeuczenia.
- **Flatten** zamienia tensor na wektor.
- **Warstwa Dense z sigmoid** zwraca pojedynczą wartość z zakresu [0,1], interpretowaną jako prawdopodobieństwo, że obraz jest prawdziwy.

---

In [ ]:
def make_discriminator_model():
    model = tf.keras.Sequential()
    model.add(layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same',
                                     input_shape=[64,64, 3]))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Flatten())
    model.add(layers.Dense(1, activation='sigmoid'))

    return model

In [ ]:
discriminator = make_discriminator_model()
decision = discriminator(generated_image)
print (decision)



### Funkcje straty

Uczenie GAN-a opiera się na grze dwuosobowej o sumie zerowej:

#### Strata dyskryminatora

Dyskryminator minimalizuje błąd klasyfikacji dla:
- obrazów prawdziwych (powinny być klasyfikowane jako 1),
- obrazów wygenerowanych (powinny być klasyfikowane jako 0).

Funkcja straty:
- `real_loss` — kara za błędną klasyfikację danych rzeczywistych,
- `fake_loss` — kara za błędną klasyfikację danych wygenerowanych,
- `total_loss` — suma obu składników.

---

#### Strata generatora

Generator nie ma bezpośredniego dostępu do prawdziwych etykiet — jego celem jest „oszukanie” dyskryminatora.

Dlatego minimalizuje funkcję straty, w której:
- wygenerowane obrazy są traktowane jak **prawdziwe** (etykieta 1).

Innymi słowy, generator uczy się generować takie próbki, które maksymalizują prawdopodobieństwo, że dyskryminator uzna je za rzeczywiste.

---

In [ ]:
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

In [ ]:
def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    total_loss = real_loss + fake_loss
    return total_loss

In [ ]:
def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

In [ ]:
generator_optimizer = tf.keras.optimizers.Adam()
discriminator_optimizer = tf.keras.optimizers.Adam()

In [ ]:
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(generator_optimizer=generator_optimizer,
                                 discriminator_optimizer=discriminator_optimizer,
                                 generator=generator,
                                 discriminator=discriminator)


### Intuicja procesu uczenia

- Na początku generator tworzy losowy szum.
- Dyskryminator łatwo rozróżnia dane prawdziwe od fałszywych.
- Z czasem generator uczy się struktur danych (np. kształtów, tekstur).
- Dyskryminator musi wykrywać coraz subtelniejsze różnice.
- Proces dąży do równowagi, w której wygenerowane obrazy są statystycznie podobne do rzeczywistych.

In [ ]:
# Ustawienie parametrów treningu
EPOCHS = 5
noise_dim = 100
num_examples_to_generate = 8
BATCH_SIZE = 64

# ustawienie globalnego ziarna dla wszystkich losowych generacji
seed = tf.random.normal([num_examples_to_generate, noise_dim])

Defninicją pojedynczego kroku treningu modelu GAN:
- stworzenie szumu
- definicja obliczania gradientu z obiektem GradientTape
- generacja obrazu
- wywołanie dyskryminatora na obrazie generowanym i orginalnym
- wyznaczenie straty dla generatora i dyskryminatora
- obliczenie i aplikacja gradientów do wag sieci

In [ ]:
@tf.function
def train_step(images):
    noise = tf.random.normal([BATCH_SIZE, noise_dim])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
      generated_images = generator(noise, training=True)

      real_output = discriminator(images, training=True)
      fake_output = discriminator(generated_images, training=True)

      gen_loss = generator_loss(fake_output)
      disc_loss = discriminator_loss(real_output, fake_output)

    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    generator_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    discriminator_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))

Funkcja pomocnicza do generacji obrazów podczas treningu

In [ ]:
def generate_and_save_images(model, epoch, test_input):
  predictions = model(test_input, training=False)

  fig = plt.figure(figsize=(4, 4))

  for i in range(predictions.shape[0]):
      plt.subplot(4, 4, i+1)
      plt.imshow(predictions[i, :, :, 0] * 127.5 + 127.5)
      plt.axis('off')

  plt.savefig('image_at_epoch_{:04d}.png'.format(epoch))
  plt.show()

Definicja treningu modelu GAN

In [ ]:
def train(dataset, epochs):
  for epoch in range(epochs):
    start = time.time()

    for image_batch in dataset:
      train_step(image_batch)

    # Save the model every 15 epochs
    if (epoch + 1) % 5 == 0:
      checkpoint.save(file_prefix = checkpoint_prefix)

    print ('Time for epoch {} is {} sec'.format(epoch + 1, time.time()-start))

  # Generate after the final epoch
  display.clear_output(wait=True)
  generate_and_save_images(generator,
                           epochs,
                           seed)
  generator.save("final_model.keras")


Trening sieci GAN. Może potrwać około 15 minut.

In [ ]:
train(train_dataset, EPOCHS)

Generacja przykładowego obrazu po treningu.

In [ ]:
test_input = tf.random.normal([1, 100])
generated_image_after_training = generator(test_input, training=False)

In [ ]:
plt.imshow(generated_image_after_training[0, :, :, 0])

Ładowanie modelu

In [ ]:
model = keras.models.load_model("final_model.keras")

**Zadanie** Zbudować sieć GAN dla danych fashion mnist (podobnie jak w powyższym przykładzie):
- wczytać i przygotować dane do treningu
- zdefiniowac strukturę sieci GAN - dyksryminator i generator. **NOTE** Istotne jest dobranie odpowiednich kształtów poszczególnych warstw sieci, tak żeby odpowiadały kształtowi obrazków wejściowych (x,y, liczba kanałów)
- dobrać odpowiednią wielkość BATCH'a
- wytrenować model GAN
- wygenerować 5 obrazów z losowo wytworzonych danych wejściowych (random noise)

In [ ]:
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.fashion_mnist.load_data()

**UWAGA** Należy uzupełnić poprawnie poniższą komórke do przygotowania danych do uczenia

In [ ]:
train_images = train_images.reshape(ilość danych, rozmiar obrazka oś x, rozmiar obrazka oś y, ilość kanałow).astype(zmiana na typ float32)

In [ ]:
BUFFER_SIZE = 60000
BATCH_SIZE = 256
train_images = # Znormalizować obrazy do zakresu [0, 1]

Stworzenie obiektu Dataset do treningu sieci GAN

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices(train_images).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)
